# TensiMenu Model 03 — K-Nearest Neighbors + RobustScaler

**Author**: Isrezal Akbar

**Pendekatan**:
- Normalisasi: **RobustScaler** (median + IQR) — tahan outlier
- Algoritma: **NearestNeighbors** dari sklearn dengan metric `cosine`
- Berbeda dari v1: query menggunakan k-nearest neighbors index (lebih efisien untuk dataset besar)

**Hipotesis**: Dataset TKPI punya banyak outlier (misal makanan tinggi natrium ekstrem seperti garam). RobustScaler menggunakan median dan IQR alih-alih mean/std, sehingga outlier tidak menggeser distribusi. NearestNeighbors juga menyediakan API yang lebih bersih untuk top-k retrieval.

---

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_BASE = '/content/drive/MyDrive/TensiMenu_ML'
    print(f'Google Drive mounted. Base: {DRIVE_BASE}')
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print('Bukan di Colab — menggunakan path lokal.')

In [ ]:
import json, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

ARTIFACTS_DIR = Path(f'{DRIVE_BASE}/artifacts_v3') if IN_COLAB else Path('artifacts_v3')
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_VERSION = '3.0.0-knn-robust'
print(f'Model: {MODEL_VERSION}')

In [ ]:
DATA_PATH = f'{DRIVE_BASE}/datasets/TKPI_2017_dataset  .csv' if IN_COLAB else '../datasets/TKPI_2017_dataset  .csv'
df_raw = pd.read_csv(DATA_PATH)

for col in ['PROTEIN_g', 'KALSIUM_mg', 'NATRIUM_mg']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.rename(columns={
    'KODE': 'food_code', 'NAMA_BAHAN': 'name', 'KATEGORI': 'category',
    'ENERGI_kal': 'energy_kcal', 'LEMAK_g': 'fat_total_g', 'SERAT_g': 'fiber_g',
    'KALSIUM_mg': 'calcium_mg', 'NATRIUM_mg': 'sodium_mg', 'KALIUM_mg': 'potassium_mg',
})

DASH_FEATURES = ['sodium_mg', 'potassium_mg', 'calcium_mg', 'fiber_g', 'fat_total_g']
RELEVANT = ['Serealia', 'Umbi Berpati', 'Kacang & Biji', 'Sayuran',
            'Buah', 'Daging & Unggas', 'Ikan, Kerang & Udang', 'Telur', 'Susu']

df = df[df['category'].isin(RELEVANT)].copy()
df = df[df[DASH_FEATURES].notna().sum(axis=1) >= 3].copy()
for f in DASH_FEATURES:
    df[f] = df.groupby('category')[f].transform(lambda s: s.fillna(s.median()))
    df[f] = df[f].fillna(df[f].median()).clip(lower=0)
df_clean = df.reset_index(drop=True)
print(f'Dataset bersih: {len(df_clean)} item')

## Deteksi Outlier (Justifikasi RobustScaler)

In [ ]:
print('=== OUTLIER DETECTION (IQR method) ===')
for feat in DASH_FEATURES:
    q1, q3 = df_clean[feat].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_outliers = ((df_clean[feat] < lower) | (df_clean[feat] > upper)).sum()
    pct = n_outliers / len(df_clean) * 100
    print(f'  {feat:15s}: {n_outliers:3d} outliers ({pct:.1f}%) | IQR=[{lower:.0f}, {upper:.0f}]')

## Normalisasi dengan RobustScaler

RobustScaler: $x_{scaled} = \frac{x - \text{median}}{\text{IQR}}$

Tahan outlier karena median dan IQR tidak terpengaruh nilai ekstrem.

In [ ]:
X = df_clean[DASH_FEATURES].to_numpy(dtype=np.float64)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print('Median per fitur:', dict(zip(DASH_FEATURES, scaler.center_.round(2))))
print('IQR per fitur:   ', dict(zip(DASH_FEATURES, scaler.scale_.round(2))))

## K-Nearest Neighbors Index

In [ ]:
TOP_K = 20
knn = NearestNeighbors(n_neighbors=TOP_K, metric='cosine', algorithm='brute')
knn.fit(X_scaled)
print(f'✓ KNN index terlatih: {len(df_clean)} item, top-K={TOP_K}, metric=cosine')

In [ ]:
def calculate_personal_targets(profile):
    w, h, age, g = profile['weight_kg'], profile['height_cm'], profile['age'], profile['gender']
    bmr = (10*w) + (6.25*h) - (5*age) + (5 if g == 'laki-laki' else -161)
    t = {'sodium_mg': 2300.0, 'potassium_mg': 4000.0,
         'calcium_mg': 1200.0 if age > 50 else 1000.0,
         'fiber_g': 38.0 if g == 'laki-laki' else 25.0,
         'fat_total_g': round(bmr * 0.27 / 9, 1)}
    if 'ckd' in profile.get('comorbidities', []):
        t['sodium_mg'] = 1500.0; t['potassium_mg'] = 2000.0
    if profile.get('systolic_bp', 0) >= 150:
        t['sodium_mg'] = 1500.0
    return t

profile = {'gender': 'laki-laki', 'weight_kg': 70, 'height_cm': 170, 'age': 45,
           'comorbidities': [], 'systolic_bp': 140}
targets = calculate_personal_targets(profile)
user_vec = np.array([targets[f] for f in DASH_FEATURES])
user_vec_scaled = scaler.transform(user_vec.reshape(1, -1))

distances, indices = knn.kneighbors(user_vec_scaled, n_neighbors=15)

results = df_clean.iloc[indices[0]].copy()
results['cosine_distance'] = distances[0]
results['similarity'] = 1 - distances[0]
print('=== TOP 15 (KNN + RobustScaler) ===')
results[['food_code', 'name', 'category', 'similarity']].reset_index(drop=True)

In [ ]:
joblib.dump(scaler, ARTIFACTS_DIR / 'scaler.pkl')
joblib.dump(knn, ARTIFACTS_DIR / 'knn_model.pkl')
np.save(ARTIFACTS_DIR / 'item_matrix.npy', X_scaled)
with open(ARTIFACTS_DIR / 'food_ids.json', 'w', encoding='utf-8') as f:
    json.dump(df_clean['food_code'].tolist(), f, ensure_ascii=False)

metadata = {
    'version': MODEL_VERSION,
    'approach': 'NearestNeighbors (cosine) + RobustScaler',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'random_state': RANDOM_STATE,
    'n_items': len(df_clean),
    'features': DASH_FEATURES,
    'scaler_class': 'RobustScaler',
    'algorithm': 'NearestNeighbors',
    'metric': 'cosine',
    'top_k': TOP_K,
}
with open(ARTIFACTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
df_clean.to_csv(ARTIFACTS_DIR / 'food_items_clean.csv', index=False)
print('✓ Artefak v3 tersimpan')